# Proyecto: Clasificación de Cerveza Premium (beer_quality_lab.csv)

Se proporciona el archivo **`beer_quality_lab.csv`**, que contiene **600 registros**.  
Cada fila representa una cerveza artesanal producida en lotes experimentales.

La variable objetivo es **`es_premium`** (`1` = cerveza premium, `0` = estándar).

## Variables del Dataset

| Variable                  | Tipo       | Descripción                                  |
|---------------------------|------------|----------------------------------------------|
| amargor_IBU               | numérico   | Nivel de amargor (0–120)                     |
| alcohol_ABV               | numérico   | Grado alcohólico (%)                          |
| color_SRM                 | numérico   | Oscuridad del color                           |
| densidad_original         | numérico   | Densidad antes de fermentación                |
| lupulo_aroma              | categórico | bajo, medio, alto                             |
| levadura                  | categórico | US05, K97, S04, BE256                         |
| tiempo_fermentacion_dias  | numérico   | Días                                          |
| carbonatacion_vol         | numérico   | CO₂ en volúmenes                              |
| es_premium                | binaria    | Variable objetivo                             |

## Instrucciones Generales

- No es necesario hacer selección de variables.  
- Solo aplica transformaciones pertinentes:
  - **One-hot encoding** para variables categóricas  
  - **Estandarización** para variables numéricas  
- **No es necesario balancear** las clases.

Debes adjuntar un archivo **`archivo.py`** con tu código y responder cada pregunta.

# Preguntas

## Pregunta 1

**Carga el dataset y separa los datos en entrenamiento (80%) y prueba (20%).**

1. Muestra cuántas observaciones hay de cada clase.

## Pregunta 2

Entrena una **regresión logística** y calcula:

- **Precision**
- **Recall**
- **F1-score** en el test

Explica **por qué F1 es o no adecuado** para accuracy en este dataset.

## Pregunta 3

Entrena un **SVC con kernel RBF**.

Realiza una **búsqueda de hiperparámetros reducida** (pocas combinaciones).

Reporta:

- Mejor combinación encontrada  
- Precision, recall, F1-score en test

## Pregunta 4

Entrena un **Random Forest**:

- Usa **validación cruzada (k=5)**
- Reporta:
  - Métricas promedio (precision, recall, F1)
- Muestra las **5 variables más importantes**

## Pregunta 5

Supón que quieres implementar este modelo en la cervecera **“Minerva”**.

Responde:

1. **¿Qué modelo elegirías y por qué?**  
2. **Explica cómo comunicarías los resultados al jefe de ventas**, resaltando los factores clave que influyen en que una cerveza sea premium.


### $\color{#dda}{\text{Clasificación de Cervezas Premium}}$ 
### $\color{#dda}{\text{Dataset: }}$ beer_quality_lab.csv
### $\color{#dda}{\text{Objetivo: }}$ Predecir si una cerveza es premium (es_premium = 1) o estándar (0)

In [17]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

## $\color{#dda}{\text{1. Carga y separación de datos}}$

In [18]:
# Cargar datos
df = pd.read_csv('beer_quality_lab.csv')

print(f"\nDimensiones del dataset: {df.shape}")
print(f"\nPrimeras filas:\n{df.head()}")
print(f"\nInformación del dataset:\n{df.info()}")


Dimensiones del dataset: (600, 9)

Primeras filas:
   amargor_IBU  alcohol_ABV  color_SRM  densidad_original lupulo_aroma  \
0    31.745423     3.747607  14.871837           1.044902        medio   
1    48.106908     6.704599   7.947215           1.070265        medio   
2    39.616689     5.886887   7.128324           1.057311        medio   
3    57.590935     4.579238   9.077901           1.052243         bajo   
4    48.154002     2.395786  14.718901           1.054810        medio   

  levadura  tiempo_fermentacion_dias  carbonatacion_vol  es_premium  
0    BE256                 10.729270           1.985522           0  
1    BE256                 26.619715           1.860536           1  
2    BE256                 12.350084           1.732928           0  
3    BE256                 11.045210           2.244579           0  
4      K97                  7.491980           2.309706           0  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data column

In [19]:
# Contar observaciones por clase
print("\n--- Distribución de clases ---")
class_distribution = df['es_premium'].value_counts().sort_index()
print(class_distribution)
print(f"\nCerveza estándar (0): {class_distribution[0]} observaciones ({class_distribution[0]/len(df)*100:.2f}%)")
print(f"Cerveza premium (1): {class_distribution[1]} observaciones ({class_distribution[1]/len(df)*100:.2f}%)")


--- Distribución de clases ---
es_premium
0    485
1    115
Name: count, dtype: int64

Cerveza estándar (0): 485 observaciones (80.83%)
Cerveza premium (1): 115 observaciones (19.17%)


In [20]:
# Separar características y variable objetivo
X = df.drop('es_premium', axis=1)
y = df['es_premium']

# One-Hot Encoding para variables categóricas
print("\n--- Aplicando One-Hot Encoding ---")
X_encoded = pd.get_dummies(X, columns=['lupulo_aroma', 'levadura'], drop_first=True)
print(f"Características después de encoding: {X_encoded.shape[1]}")
print(f"Columnas: {list(X_encoded.columns)}")


--- Aplicando One-Hot Encoding ---
Características después de encoding: 11
Columnas: ['amargor_IBU', 'alcohol_ABV', 'color_SRM', 'densidad_original', 'tiempo_fermentacion_dias', 'carbonatacion_vol', 'lupulo_aroma_bajo', 'lupulo_aroma_medio', 'levadura_K97', 'levadura_S04', 'levadura_US05']


In [21]:
# Separar en train y test (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nConjunto de entrenamiento: {X_train.shape[0]} observaciones")
print(f"Conjunto de prueba: {X_test.shape[0]} observaciones")


Conjunto de entrenamiento: 480 observaciones
Conjunto de prueba: 120 observaciones


In [22]:
# Estandarizar características numéricas
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## $\color{#dda}{\text{2. Regresión Logística}}$

In [23]:
# Entrenar modelo
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

# Predicciones
y_pred_log = log_reg.predict(X_test_scaled)

# Métricas
precision_log = precision_score(y_test, y_pred_log)
recall_log = recall_score(y_test, y_pred_log)
f1_log = f1_score(y_test, y_pred_log)

print("\n--- Métricas en conjunto de prueba ---")
print(f"Precision: {precision_log:.4f}")
print(f"Recall: {recall_log:.4f}")
print(f"F1-Score: {f1_log:.4f}")


--- Métricas en conjunto de prueba ---
Precision: 0.9167
Recall: 0.9565
F1-Score: 0.9362


In [24]:
print("\n--- Reporte de clasificación completo ---")
print(classification_report(y_test, y_pred_log, target_names=['Estándar', 'Premium']))


--- Reporte de clasificación completo ---
              precision    recall  f1-score   support

    Estándar       0.99      0.98      0.98        97
     Premium       0.92      0.96      0.94        23

    accuracy                           0.97       120
   macro avg       0.95      0.97      0.96       120
weighted avg       0.98      0.97      0.98       120



### ¿Por qué F1-Score es más adecuado que Accuracy?
RESPUESTA:
F1-Score es más adecuado que Accuracy para este dataset por las siguientes razones:

1. DESBALANCE DE CLASES: El dataset puede tener un desbalance entre cervezas
   premium y estándar. Accuracy puede ser engañoso cuando hay desbalance, ya que
   un modelo que siempre prediga la clase mayoritaria tendría alta accuracy pero
   sería inútil en la práctica.

2. BALANCE ENTRE PRECISION Y RECALL: F1-Score es la media armónica entre precision
   y recall, lo que significa que penaliza modelos con desbalance entre estas métricas.
   Esto es importante porque:
   - Precision baja: Etiquetar cervezas estándar como premium (costo económico)
   - Recall bajo: No identificar cervezas premium (pérdida de oportunidad de mercado)

3. COSTOS ASIMÉTRICOS: En el negocio cervecero, clasificar incorrectamente una
   cerveza como premium cuando no lo es, o viceversa, tiene implicaciones económicas
   y de reputación. F1-Score captura mejor estos errores que accuracy.

4. EVALUACIÓN MÁS CONSERVADORA: F1-Score da una evaluación más estricta y realista
   del rendimiento del modelo, especialmente útil para la clase minoritaria (premium).

## $\color{#dda}{\text{3. SVM con kernel RBF}}$

In [25]:
# Búsqueda de hiperparámetros reducida
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': [0.01, 0.1, 1]
}

svm_model = SVC(kernel='rbf', random_state=42)
grid_search = GridSearchCV(
    svm_model, 
    param_grid, 
    cv=3, 
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

print(f"\nMejor combinación de hiperparámetros:")
print(f"  C = {grid_search.best_params_['C']}")
print(f"  gamma = {grid_search.best_params_['gamma']}")
print(f"  F1-Score (validación cruzada): {grid_search.best_score_:.4f}")


Mejor combinación de hiperparámetros:
  C = 10
  gamma = 0.01
  F1-Score (validación cruzada): 0.8657


In [26]:
# Evaluar en test
best_svm = grid_search.best_estimator_
y_pred_svm = best_svm.predict(X_test_scaled)

precision_svm = precision_score(y_test, y_pred_svm)
recall_svm = recall_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm)

print("\n--- Métricas en conjunto de prueba ---")
print(f"Precision: {precision_svm:.4f}")
print(f"Recall: {recall_svm:.4f}")
print(f"F1-Score: {f1_svm:.4f}")

print("\n--- Reporte de clasificación completo ---")
print(classification_report(y_test, y_pred_svm, target_names=['Estándar', 'Premium']))


--- Métricas en conjunto de prueba ---
Precision: 0.9130
Recall: 0.9130
F1-Score: 0.9130

--- Reporte de clasificación completo ---
              precision    recall  f1-score   support

    Estándar       0.98      0.98      0.98        97
     Premium       0.91      0.91      0.91        23

    accuracy                           0.97       120
   macro avg       0.95      0.95      0.95       120
weighted avg       0.97      0.97      0.97       120



## $\color{#dda}{\text{4. Random Forest con validación cruzada}}$

In [27]:
# Entrenar Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# Validación cruzada k=5
print("\n--- Validación cruzada (k=5) ---")
cv_results = cross_validate(
    rf_model, 
    X_train_scaled, 
    y_train,
    cv=5,
    scoring=['precision', 'recall', 'f1'],
    return_train_score=False
)

print(f"\nPrecision promedio: {cv_results['test_precision'].mean():.4f} ± {cv_results['test_precision'].std():.4f}")
print(f"Recall promedio: {cv_results['test_recall'].mean():.4f} ± {cv_results['test_recall'].std():.4f}")
print(f"F1-Score promedio: {cv_results['test_f1'].mean():.4f} ± {cv_results['test_f1'].std():.4f}")


--- Validación cruzada (k=5) ---

Precision promedio: 0.9028 ± 0.0342
Recall promedio: 0.7731 ± 0.1035
F1-Score promedio: 0.8278 ± 0.0504


In [28]:
# Entrenar en todo el conjunto de entrenamiento para obtener importancias
rf_model.fit(X_train_scaled, y_train)

# Evaluar en test
y_pred_rf = rf_model.predict(X_test_scaled)
print("\n--- Métricas en conjunto de prueba ---")
print(f"Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_rf):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_rf):.4f}")


--- Métricas en conjunto de prueba ---
Precision: 0.8800
Recall: 0.9565
F1-Score: 0.9167


## $\color{#dda}{\text{5. Recomendación y comunicación de resultados}}$

In [29]:
# Importancia de variables
feature_importance = pd.DataFrame({
    'feature': X_encoded.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n--- Top 10 variables más importantes ---")
print(feature_importance.head(10))


--- Top 10 variables más importantes ---
                     feature  importance
3          densidad_original    0.376870
1                alcohol_ABV    0.283746
0                amargor_IBU    0.118330
4   tiempo_fermentacion_dias    0.093707
5          carbonatacion_vol    0.049601
2                  color_SRM    0.046283
7         lupulo_aroma_medio    0.009895
10             levadura_US05    0.006063
8               levadura_K97    0.006053
6          lupulo_aroma_bajo    0.004864


In [31]:
# Comparación de modelos
print("\n--- COMPARACIÓN DE MODELOS ---")
print(f"\n{'Modelo':<25} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
print("-" * 65)
print(f"{'Regresión Logística':<25} {precision_log:<12.4f} {recall_log:<12.4f} {f1_log:<12.4f}")
print(f"{'SVM (RBF)':<25} {precision_svm:<12.4f} {recall_svm:<12.4f} {f1_svm:<12.4f}")
print(f"{'Random Forest':<25} {precision_score(y_test, y_pred_rf):<12.4f} {recall_score(y_test, y_pred_rf):<12.4f} {f1_score(y_test, y_pred_rf):<12.4f}")



--- COMPARACIÓN DE MODELOS ---

Modelo                    Precision    Recall       F1-Score    
-----------------------------------------------------------------
Regresión Logística       0.9167       0.9565       0.9362      
SVM (RBF)                 0.9130       0.9130       0.9130      
Random Forest             0.8800       0.9565       0.9167      


### $\color{#dda}{\text{MODELO ELEGIDO: }}$ Regresión Logística

**JUSTIFICACIÓN TÉCNICA:**

**MEJOR RENDIMIENTO GENERAL:** Regresión Logística muestra las métricas más altas en las tres métricas clave:
Precision: 91.67% (la más alta)
Recall: 95.65% (la más alta, empatada con RF)
F1-Score: 93.62% (la más alta)

**BALANCE ÓPTIMO**: Logra el mejor equilibrio entre precision y recall, lo que significa que identifica correctamente la mayoría de cervezas premium (95.65%) mientras mantiene un bajo índice de falsos positivos.

**SIMPLICIDAD E INTERPRETABILIDAD:** Es un modelo más simple que Random Forest y SVM, lo que facilita:
Explicación a stakeholders no técnicos
Debugging y mantenimiento
Auditoría de decisiones

**EFICIENCIA COMPUTACIONAL:** Requiere menos recursos para entrenamiento y predicción que Random Forest, lo que es crucial para implementación en producción y escalabilidad.

**MENOR RIESGO DE OVERFITTING:** Su simplicidad reduce el riesgo de sobreajuste comparado con Random Forest, garantizando mejor generalización con datos nuevos.

**VELOCIDAD DE PREDICCIÓN:** Las predicciones son prácticamente instantáneas, ideal para integración en sistemas de producción en tiempo real.

### $\color{#dda}{\text{E-Mail para el patrón del depto de ventas}}$

#### **Asunto:** Modelo predictivo para identificación de cervezas premium | Departamento de BI
=====================================

Buenas tardes,

Con el propósito de comunicarle al departamento de ventas que en @BI hemos desarrollado un modelo predictivo que identifica con alta precisión qué cervezas tienen características de producto premium.

A través del modelo pudimos deducir los **5 factores críticos** que influyen en la calidad de una cerveza:
1. **DENSIDAD ORIGINAL (37.7% de importancia):** Es el factor dominante. Una mayor densidad inicial indica más azúcares disponibles para fermentación, resultando en cervezas con mayor cuerpo, complejidad y calidad percibida.
2. **CONTENIDO ALCOHÓLICO - ABV (28.4%):** Las cervezas premium tienen graduaciones alcohólicas más altas, lo que se traduce en mayor cuerpo, sabor complejo y percepción de valor.
3. **AMARGOR - IBU (11.8%):** El nivel de amargor es un diferenciador clave. Las cervezas premium tienen perfiles de amargor más sofisticados y balanceados.
4. **TIEMPO DE FERMENTACIÓN (9.4%):** Los procesos más largos desarrollan sabores más complejos, matices aromáticos y perfiles organolépticos superiores.
5. **CARBONATACIÓN (5.0%):** El nivel adecuado de CO₂ afecta la textura, sensación en boca y experiencia sensorial general.

Y como se imaginarán, esto tiene diversas utilidades:
- **CONTROL DE CALIDAD:** Predecir si un lote alcanzará estándares premium antes
  de completar producción.
- **OPTIMIZACIÓN DE RECETAS:** Ajustar variables clave para elevar cervezas
  estándar a premium.
- **ESTRATEGIA DE PRECIOS:** Justificar precios premium basados en características
  medibles y objetivas.
- **DESARROLLO DE PRODUCTOS:** Diseñar nuevas recetas con características premium
  desde el inicio.

Gracias de antemano y feliz fin de semana.